# CTI-VSP walkthrough — CVE description → CVSS vector

**Task:** predict the CVSS v3.1 base vector for a CVE.
**Scoring:** `1 − MAD/7.7` over base scores (athenabench), plus severity-band agreement and
component distance. **Cadence:** hourly.

In [17]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

repo root : /Users/jake/Projects/glokta
model     : huggingface/meta-llama/Llama-3.1-8B-Instruct
LIVE calls: True (set HF_TOKEN to enable real inference)


## 1. Dataflow — the VSP item carries the CVSS label

In [18]:
# A CVE JSON 5.0 record (cvelistV5 shape). In production these come from the delta feed
# (fetch_recent_cve_records); here we use a representative in-memory example.
CVE_RECORD = {
    "cveMetadata": {"cveId": "CVE-2024-12345", "datePublished": "2024-05-01T10:00:00.000Z",
                    "dateUpdated": "2024-05-20T10:00:00.000Z", "state": "PUBLISHED"},
    "containers": {
        "cna": {
            "descriptions": [{"lang": "en",
                "value": "A SQL injection vulnerability in Acme Portal allows a remote "
                         "unauthenticated attacker to execute arbitrary SQL via the search parameter."}],
            "problemTypes": [{"descriptions": [{"lang": "en", "cweId": "CWE-89",
                              "description": "SQL Injection"}]}],
            "metrics": [{"cvssV3_1": {"vectorString": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
                                       "baseScore": 9.8}}],
        },
        # CISA-ADP (Vulnrichment) container — here it agrees with the CNA on the CWE.
        "adp": [{"providerMetadata": {"shortName": "CISA-ADP", "dateUpdated": "2024-05-20T10:00:00.000Z"},
                 "problemTypes": [{"descriptions": [{"lang": "en", "cweId": "CWE-89"}]}]}],
    },
}

In [19]:
from glokta.infrastructure.cti.connectors.cve import normalise_cve_record

vsp = next(i for i in normalise_cve_record(CVE_RECORD, "walkthrough") if i.task == "vsp")
print("input_text:", vsp.input_text[:90], "...")
print("label     :", vsp.label)   # {'vector': 'CVSS:3.1/...', 'base_score': 9.8}

input_text: A SQL injection vulnerability in Acme Portal allows a remote unauthenticated attacker to e ...
label     : {'vector': 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H', 'base_score': 9.8}


## 2. Tasking — prompt + model call

In [20]:
from glokta.infrastructure.cti.prompts import build_prompt, parse_response

prompt = build_prompt("vsp", vsp.input_text)
print(prompt)
print("-" * 70)
response = run_model(prompt, canned="Answer: CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H")
print("model response:", repr(response))
print("parsed vector :", parse_response("vsp", response))

You are a vulnerability analyst. Given the CVE description below, predict the CVSS v3.1 base vector string.

CVE description:
A SQL injection vulnerability in Acme Portal allows a remote unauthenticated attacker to execute arbitrary SQL via the search parameter.

Respond with only the CVSS v3.1 vector, e.g. CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H.
Answer:
----------------------------------------------------------------------
model response: 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H'
parsed vector : CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H


## 3. Scoring — base-score MAD + severity + component distance

In [21]:
from glokta.infrastructure.cti.evaluator import evaluate_item
from glokta.domain.cti.scoring import score_vsp

scored = evaluate_item("vsp", vsp.label, response)
print("score (1 - MAD/7.7):", scored.score)
print("correct (severity match):", scored.correct)
print("breakdown:", scored.breakdown)

score (1 - MAD/7.7): 1.0
correct (severity match): True
breakdown: {'pred_base': 9.8, 'label_base': 9.8, 'mad': 0.0, 'severity_match': True, 'component_distance': 0.0}


## 4. A handful of examples
From exact to a low-severity guess — watch MAD and severity_match move.

In [22]:
candidates = {
    "exact":        "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
    "near (AV:L)":  "CVSS:3.1/AV:L/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
    "low severity": "CVSS:3.1/AV:N/AC:H/PR:H/UI:R/S:U/C:N/I:N/A:L",
    "unparseable":  "I am not sure of the score.",
}
for name, vec in candidates.items():
    s, bd = score_vsp(vec if vec.startswith('CVSS') else None, vsp.label['vector'])
    print(f"{name:13} score={s:.3f} mad={bd['mad']:.2f} sev_match={bd['severity_match']} comp_dist={bd['component_distance']:.3f}")

exact         score=1.000 mad=0.00 sev_match=True comp_dist=0.000
near (AV:L)   score=0.818 mad=1.40 sev_match=False comp_dist=0.125
low severity  score=0.000 mad=7.80 sev_match=False comp_dist=0.750
unparseable   score=0.000 mad=9.80 sev_match=False comp_dist=1.000


**Takeaway:** VSP rewards getting the *severity right* even when the exact vector differs — MAD shrinks as the predicted base score nears the label, and `component_distance` shows how many base metrics were wrong.